# Exploratory Data Analysis - Telco Customer Churn

Goal: understand **who leaves** and **why**, before modelling.

Dataset: IBM Telco Customer Churn (7,043 customers, 21 columns). Run `make data` first.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import seaborn as sns

from churn.data import load_raw, clean
from churn.features import add_features

sns.set_theme(style='whitegrid')
df = add_features(clean(load_raw('../data/raw/Telco-Customer-Churn.csv')))
df.shape

## 1. Data quality

In [ ]:
df.info()
df.isna().sum().sort_values(ascending=False).head()

## 2. Target: how unbalanced is churn?

In [ ]:
ax = df['Churn'].value_counts(normalize=True).rename({0: 'Stay', 1: 'Churn'}).plot.bar(color=['#4c72b0', '#dd8452'])
ax.set_title('Churn rate'); ax.set_ylabel('share of customers')
df['Churn'].mean()

Churn is a minority class (~26%): accuracy is misleading, so we use **ROC-AUC, PR-AUC, precision and recall**, and class weights during training.

## 3. Churn drivers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['Contract', 'InternetService', 'PaymentMethod']):
    df.groupby(col)['Churn'].mean().sort_values().plot.barh(ax=ax)
    ax.set_title(f'Churn rate by {col}')
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.kdeplot(data=df, x='tenure', hue='Churn', common_norm=False, ax=axes[0])
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', common_norm=False, ax=axes[1])
axes[0].set_title('Tenure'); axes[1].set_title('Monthly charges')

In [ ]:
df.groupby('tenure_group')['Churn'].mean().reindex(['0-6m', '6-12m', '1-2y', '2-4y', '4y+']).plot.bar(title='Churn rate by tenure group')

## 4. Correlations of numeric features with churn

In [ ]:
num = ['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'num_services',
       'charges_per_service', 'is_month_to_month', 'is_fiber', 'auto_payment', 'is_new_customer', 'Churn']
plt.figure(figsize=(9, 7))
sns.heatmap(df[num].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)

## 5. Key takeaways (fill in after running)

- Month-to-month contracts churn much more than 1-2 year contracts.
- New customers (first 6 months) are the most at risk.
- Fiber optic and electronic check payment are associated with higher churn.

These insights drive the engineered features (`is_month_to_month`, `is_new_customer`, `is_fiber`, `auto_payment`) and the business-rule baseline used in `churn.train`.